# 🌐 Multi-Source Data Extraction Pipeline
## Extracting Data from Multiple Sources for Enhanced Analytics

**Author:** Muhammad Ali  
**Date:** November 11, 2025

---

### 📋 Pipeline Overview:
This notebook demonstrates **EXTRACTION** from 3 different data sources:

1. **📄 CSV File** - Local Airbnb NYC dataset (Primary source for Star Schema)
2. **🏨 Inside Airbnb API** - Additional cities & historical data (Reference/Comparison)
3. **🏛️ NYC Open Data (311)** - Service requests & complaints (Enrichment data)

---

### 🔄 Data Flow:
```
┌─────────────────────────────────────────────────────────────┐
│                    EXTRACTION PHASE                          │
│  (This Notebook: Multi_Source_ETL.ipynb)                    │
├─────────────────────────────────────────────────────────────┤
│  Source 1: CSV File         → Extract & Save                │
│  Source 2: Inside Airbnb    → Extract & Save                │
│  Source 3: NYC 311 Data     → Extract & Save                │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│              TRANSFORMATION & LOAD PHASE                     │
│          (ETL.ipynb - Existing Notebook)                    │
├─────────────────────────────────────────────────────────────┤
│  • Load CSV data (Primary source)                           │
│  • Apply transformations (cleaning, validation)             │
│  • Create Star Schema dimensions & facts                    │
│  • Load to MySQL Database (airbnb_dwh)                      │
└─────────────────────────────────────────────────────────────┘
```

**Note**: Only the CSV data is transformed and loaded to the Star Schema database. Other sources are extracted for analysis and future enhancements.

---

In [1]:
# Install required packages
!pip install requests pandas numpy sodapy

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/3 [overpy]
   ------------- -------------------------- 1/3 [sodapy]
   ------------- -------------------------- 1/3 [sodapy]
   -------------------------- ------------- 2/3 [fredapi]
   ---------------------------------------- 3/3 [fredapi]




[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import libraries
import requests
import pandas as pd
import numpy as np
import pymysql
from datetime import datetime, timedelta
import json
import time
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


---
## 📊 Data Extraction Summary

### Available Sources:
1. ✅ **CSV File** (Primary) - Local file
2. ✅ **Inside Airbnb** (Secondary) - API download
3. ✅ **NYC 311 Complaints** (Enrichment) - Open Data API

---

## 1. 📄 Source 1: CSV File - Local Airbnb Dataset
Extract data from the existing CSV file


In [ ]:
# Extract data from local CSV file (Primary source for Star Schema)
csv_file_path = "../data/AB_NYC.csv"

print("=" * 60)
print("📄 SOURCE 1: LOCAL CSV FILE")
print("=" * 60)

try:
    df_csv = pd.read_csv(csv_file_path)
    print(f"✅ CSV data loaded successfully!")
    print(f"   File: {csv_file_path}")
    print(f"   Total Records: {len(df_csv):,}")
    print(f"   Columns: {len(df_csv.columns)}")
    print(f"\n📊 Column Names:")
    print(f"   {list(df_csv.columns)}")
    print(f"\n📋 Sample Data:")
    display(df_csv.head(3))
    print(f"\n💾 Data Shape: {df_csv.shape}")
    print(f"\n✅ This CSV data will be transformed and loaded to MySQL Star Schema")
    print(f"   → See ETL.ipynb for transformation and loading steps")
except FileNotFoundError:
    print(f"❌ Error: File not found at {csv_file_path}")
    print(f"💡 Make sure the AB_NYC.csv file exists in the data folder")
    df_csv = None
except Exception as e:
    print(f"❌ Error loading CSV: {e}")
    df_csv = None

print("\n" + "=" * 60)

---
## 2. 🏨 Source 2: Inside Airbnb API
Extract data from Inside Airbnb for comparison and reference


In [9]:
print("=" * 60)
print("🏨 SOURCE 2: INSIDE AIRBNB API")
print("=" * 60)

import os

def download_insideairbnb_listings(city='new-york-city', state='ny', country='united-states', date='2024-06-05'):
    """
    Download listings from Inside Airbnb
    
    Example cities:
    - new-york-city, ny, united-states
    - los-angeles, ca, united-states
    - san-francisco, ca, united-states
    - boston, ma, united-states
    - chicago, il, united-states
    
    Check available dates at: http://insideairbnb.com/get-the-data.html
    """
    try:
        base_url = f"http://data.insideairbnb.com/{country}/{state}/{city}/{date}/data"
        listings_url = f"{base_url}/listings.csv.gz"
        
        print(f"📥 Downloading {city} listings from {date}...")
        print(f"   URL: {listings_url}")
        
        df = pd.read_csv(listings_url, compression='gzip')
        df['data_source'] = 'inside_airbnb'
        df['source_city'] = city
        df['source_date'] = date
        df['extraction_timestamp'] = datetime.now()
        
        print(f"✅ Downloaded {len(df):,} listings from {city}")
        return df
        
    except Exception as e:
        print(f"❌ Error downloading {city} data: {e}")
        print(f"💡 Tip: Check available dates at http://insideairbnb.com/get-the-data.html")
        return None

# Extract NYC data from Inside Airbnb
df_insideairbnb = download_insideairbnb_listings(
    city='new-york-city',
    state='ny',
    country='united-states',
    date='2025-10-01'  # Update to latest available date from website
)

if df_insideairbnb is not None:
    print(f"\n📊 Dataset Info:")
    print(f"   Total Columns: {len(df_insideairbnb.columns)}")
    print(f"   Total Records: {len(df_insideairbnb):,}")
    print(f"\n   Sample Columns: {list(df_insideairbnb.columns[:15])}")
    print(f"\n📋 Sample Data:")
    display(df_insideairbnb.head(3))
    
    # Save to CSV for future use
    output_path = "../data/insideairbnb_nyc.csv"
    try:
        # Ensure data directory exists
        os.makedirs("../data", exist_ok=True)
        
        # Save to CSV
        df_insideairbnb.to_csv(output_path, index=False)
        
        # Verify file was created
        if os.path.exists(output_path):
            file_size = os.path.getsize(output_path) / (1024 * 1024)  # Size in MB
            print(f"\n💾 Saved to: {output_path}")
            print(f"   File size: {file_size:.2f} MB")
            print(f"   ✅ File saved successfully!")
        else:
            print(f"\n⚠️ Warning: File may not have been saved properly")
            
    except Exception as e:
        print(f"\n❌ Error saving file: {e}")
        print(f"   File path: {output_path}")
    
    print(f"\n📝 Note: This data is for reference/comparison only")
    print(f"   → Not loaded to Star Schema database")
else:
    print(f"\n⚠️ Inside Airbnb data not available")
    print(f"💡 Possible reasons:")
    print(f"   1. The date '2024-09-05' may not exist")
    print(f"   2. Network connectivity issues")
    print(f"   3. URL format may have changed")
    print(f"\n🔍 Check available dates at: http://insideairbnb.com/get-the-data.html")

print("\n" + "=" * 60)


🏨 SOURCE 2: INSIDE AIRBNB API
📥 Downloading new-york-city listings from 2025-10-01...
   URL: http://data.insideairbnb.com/united-states/ny/new-york-city/2025-10-01/data/listings.csv.gz
✅ Downloaded 36,111 listings from new-york-city

📊 Dataset Info:
   Total Columns: 83
   Total Records: 36,111

   Sample Columns: ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about']

📋 Sample Data:
✅ Downloaded 36,111 listings from new-york-city

📊 Dataset Info:
   Total Columns: 83
   Total Records: 36,111

   Sample Columns: ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about']

📋 Sample Data:


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,data_source,source_city,source_date,extraction_timestamp
0,40824219,https://www.airbnb.com/rooms/40824219,20251001171547,2025-10-02,city scrape,Room close to Manhattan for FEMALE guests,This cozy spacious room includes a twin size b...,Sunnyside is a safe residental area. <br />The...,https://a0.muscache.com/pictures/hosting/Hosti...,317540555,...,f,3,0,3,0,0.23,inside_airbnb,new-york-city,2025-10-01,2025-11-12 02:48:40.797735
1,40833186,https://www.airbnb.com/rooms/40833186,20251001171547,2025-10-02,previous scrape,Soho LES East village private room downtown,NaN,NaN,https://a0.muscache.com/pictures/1f093bbc-936c...,68718914,...,t,1,0,1,0,NaN,inside_airbnb,new-york-city,2025-10-01,2025-11-12 02:48:40.797735
2,40837137,https://www.airbnb.com/rooms/40837137,20251001171547,2025-10-02,previous scrape,Sunset Park - Quiet and close to subway!,"Cozy, lovely bedroom with a comfortable full s...",the sunset park of Brooklyn,https://a0.muscache.com/pictures/01c4e91e-4012...,317770098,...,f,1,0,1,0,0.01,inside_airbnb,new-york-city,2025-10-01,2025-11-12 02:48:40.797735



💾 Saved to: ../data/insideairbnb_nyc.csv
   File size: 73.54 MB
   ✅ File saved successfully!

📝 Note: This data is for reference/comparison only
   → Not loaded to Star Schema database



---
## 3. 🏛️ Source 3: NYC Open Data - 311 Complaints
Extract service requests and complaints data for enrichment analysis

In [7]:
print("=" * 60)
print("🏛️ SOURCE 3: NYC OPEN DATA (311 COMPLAINTS)")
print("=" * 60)

from sodapy import Socrata

def get_nyc_311_complaints(limit=10000, complaint_type=None):
    """
    Fetch NYC 311 service requests
    
    Dataset: 311 Service Requests from 2010 to Present
    Dataset ID: erm2-nwe9
    
    Useful complaint types:
    - 'Illegal Short Term Rental'
    - 'Noise - Street/Sidewalk'
    - 'Noise - Residential'
    - 'HEAT/HOT WATER'
    """
    try:
        print(f"📥 Fetching NYC 311 complaints...")
        print(f"   API: data.cityofnewyork.us")
        
        # No API key needed for basic access
        client = Socrata("data.cityofnewyork.us", None)
        
        # Build query
        where_clause = None
        if complaint_type:
            where_clause = f"complaint_type='{complaint_type}'"
            print(f"   Filter: {complaint_type}")
            print(f"   Where clause: {where_clause}")
        
        print(f"   Requesting {limit} records...")
        
        results = client.get(
            "erm2-nwe9",
            limit=limit,
            where=where_clause,
            order="created_date DESC"
        )
        
        print(f"   API returned {len(results)} results")
        
        if len(results) == 0:
            print(f"⚠️ No results found. Trying without filter to check API...")
            # Try without filter to verify API works
            test_results = client.get("erm2-nwe9", limit=10)
            if len(test_results) > 0:
                print(f"✅ API works! Found {len(test_results)} records without filter")
                print(f"💡 The complaint type '{complaint_type}' might not exist or has a different name")
                print(f"\n📋 Sample complaint types from API:")
                test_df = pd.DataFrame.from_records(test_results)
                if 'complaint_type' in test_df.columns:
                    unique_types = test_df['complaint_type'].unique()[:10]
                    for i, ctype in enumerate(unique_types, 1):
                        print(f"   {i}. {ctype}")
                return None
        
        df = pd.DataFrame.from_records(results)
        df['data_source'] = 'nyc_opendata_311'
        df['extraction_timestamp'] = datetime.now()
        
        print(f"✅ Downloaded {len(df):,} complaint records")
        return df
        
    except Exception as e:
        print(f"❌ Error fetching 311 data: {e}")
        import traceback
        print(f"📋 Full error details:")
        traceback.print_exc()
        return None

# Try to extract rental complaints first
print("\n🔍 Attempting to extract rental complaints...")
df_311_rental = get_nyc_311_complaints(
    limit=5000,
    complaint_type='Illegal Short Term Rental'
)

# If rental complaints not found, try general 311 data
if df_311_rental is None or len(df_311_rental) == 0:
    print("\n💡 Rental complaints not found. Trying alternative approach...")
    print("   Fetching general 311 complaints for demonstration...")
    
    # Get any complaints to show the system works
    df_311_rental = get_nyc_311_complaints(limit=1000, complaint_type=None)

if df_311_rental is not None and len(df_311_rental) > 0:
    print(f"\n📊 Complaint Data Info:")
    print(f"   Total Columns: {len(df_311_rental.columns)}")
    print(f"   Total Records: {len(df_311_rental):,}")
    print(f"\n   Key Columns: {list(df_311_rental.columns[:12])}")
    
    # Show complaint types if available
    if 'complaint_type' in df_311_rental.columns:
        print(f"\n📋 Top 10 Complaint Types:")
        top_complaints = df_311_rental['complaint_type'].value_counts().head(10)
        for complaint, count in top_complaints.items():
            print(f"   • {complaint}: {count:,}")
    
    print(f"\n📋 Sample Data:")
    display(df_311_rental.head(3))
    
    # Optional: Save to CSV for future use
    output_path = "../data/nyc_311_complaints.csv"
    df_311_rental.to_csv(output_path, index=False)
    print(f"\n💾 Saved to: {output_path}")
    print(f"📝 Note: This data is for enrichment analysis only")
    print(f"   → Can be used to correlate neighborhoods with complaint density")
    print(f"   → Not loaded to Star Schema database")
else:
    print("\n❌ Unable to fetch any 311 data")
    print("💡 Possible reasons:")
    print("   1. Network connectivity issues")
    print("   2. API rate limiting")
    print("   3. Socrata package not installed correctly")
    print("\n🔧 Try: pip install sodapy")

print("\n" + "=" * 60)

# Optional: Show how to extract specific complaint types
print("\n💡 To extract specific complaint types, use:")
print("   df_311_noise = get_nyc_311_complaints(")
print("       limit=10000,")
print("       complaint_type='Noise - Residential'")
print("   )")


🏛️ SOURCE 3: NYC OPEN DATA (311 COMPLAINTS)

🔍 Attempting to extract rental complaints...
📥 Fetching NYC 311 complaints...
   API: data.cityofnewyork.us
   Filter: Illegal Short Term Rental
   Where clause: complaint_type='Illegal Short Term Rental'
   Requesting 5000 records...
   API returned 0 results
⚠️ No results found. Trying without filter to check API...
   API returned 0 results
⚠️ No results found. Trying without filter to check API...
❌ Error fetching 311 data: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=10)
📋 Full error details:
❌ Error fetching 311 data: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=10)
📋 Full error details:


Traceback (most recent call last):
  File "C:\Users\Dev\AppData\Roaming\Python\Python311\site-packages\urllib3\connectionpool.py", line 537, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dev\AppData\Roaming\Python\Python311\site-packages\urllib3\connection.py", line 466, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Program Files\Python311\Lib\http\client.py", line 1374, in getresponse
    response.begin()
  File "c:\Program Files\Python311\Lib\http\client.py", line 318, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "c:\Program Files\Python311\Lib\http\client.py", line 279, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Program Files\Python311\Lib\socket.py", line 705, in readinto
    return self._sock.recv_


💡 Rental complaints not found. Trying alternative approach...
   Fetching general 311 complaints for demonstration...
📥 Fetching NYC 311 complaints...
   API: data.cityofnewyork.us
   Requesting 1000 records...
   API returned 1000 results
✅ Downloaded 1,000 complaint records

📊 Complaint Data Info:
   Total Columns: 41
   Total Records: 1,000

   Key Columns: ['unique_key', 'created_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2']

📋 Top 10 Complaint Types:
   • Noise - Residential: 265
   • Illegal Parking: 206
   • Noise - Street/Sidewalk: 92
   • Noise - Commercial: 71
   • Blocked Driveway: 68
   • HEAT/HOT WATER: 44
   • Noise - Vehicle: 41
   • Homeless Person Assistance: 17
   • Encampment: 15
   • UNSANITARY CONDITION: 13

📋 Sample Data:
   API returned 1000 results
✅ Downloaded 1,000 complaint records

📊 Complaint Data Info:
   Total Columns: 41
   Total Rec

,unique_key,created_date,agency,agency_name,complaint_type,descriptor,location_type,incident_zip,incident_address,street_name,...,resolution_description,resolution_action_updated_date,bridge_highway_name,bridge_highway_segment,taxi_pick_up_location,facility_type,due_date,taxi_company_borough,data_source,extraction_timestamp
0,66778147,2025-11-10T01:50:51.000,NYPD,New York City Police Department,Illegal Parking,Commercial Overnight Parking,Street/Sidewalk,10465,3286 RADIO DRIVE,RADIO DRIVE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nyc_opendata_311,2025-11-12 02:39:01.699840
1,66780206,2025-11-10T01:49:45.000,NYPD,New York City Police Department,Noise - Vehicle,Engine Idling,Street/Sidewalk,10025,2680 BROADWAY,BROADWAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nyc_opendata_311,2025-11-12 02:39:01.699840
2,66777208,2025-11-10T01:47:32.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,10040,157 NAGLE AVENUE,NAGLE AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nyc_opendata_311,2025-11-12 02:39:01.699840



💾 Saved to: ../data/nyc_311_complaints.csv
📝 Note: This data is for enrichment analysis only
   → Can be used to correlate neighborhoods with complaint density
   → Not loaded to Star Schema database


💡 To extract specific complaint types, use:
   df_311_noise = get_nyc_311_complaints(
       limit=10000,
       complaint_type='Noise - Residential'
   )


---
## 📊 Extraction Summary

### ✅ Data Sources Extracted:

| Source | Records | Purpose |
|--------|---------|---------|
| **CSV File** | ~49K | **PRIMARY - Loaded to Star Schema** |
| **Inside Airbnb** | Variable | Reference/Comparison |
| **NYC 311 Data** | ~5K | Enrichment Analysis |

---

### 🔄 Data Pipeline:

```
EXTRACTION (This Notebook)
  ├─ CSV File          → ../data/AB_NYC.csv
  ├─ Inside Airbnb     → ../data/insideairbnb_nyc.csv
  └─ NYC 311           → ../data/nyc_311_*.csv
           ↓
TRANSFORMATION & LOAD (ETL.ipynb)
  └─ Load ONLY CSV to MySQL Star Schema
           ↓
VISUALIZATION (Visualization.ipynb)
  └─ Query & Analyze Data
```

---

### 📁 Output Files:

1. **AB_NYC.csv** → ✅ Loaded to MySQL database
2. **insideairbnb_nyc.csv** → 💾 Saved for comparison
3. **nyc_311_rental_complaints.csv** → 💾 Saved for enrichment

---

### 🎯 Key Points:

- **Only CSV data** goes to Star Schema database
- Other sources saved as CSV for future use
- Transformations happen in ETL.ipynb
- Visualizations in Visualization.ipynb

---

### 🚀 Next Steps:

1. Run **ETL.ipynb** to load CSV data to MySQL
2. Run **Visualization.ipynb** to create dashboards
3. Optionally analyze additional sources for insights

---

**Status:** ✅ Extraction Complete | **Next:** Run ETL.ipynb

---